In [1]:
import json
from collections import Counter

import numpy as np
import pandas as pd

DATA_PATH = "cleaned_dataset.csv"
LOGREG_PARAMS_PATH = "logreg_params.npz"
LOGREG_TFIDF_PATH = "logreg_tfidf.json"
NB_PARAMS_PATH = "bernoulli_params.npz"
NB_VOCAB_PATH = "bernoulli_vocab.json"
STACK_META_PATH = "stack_meta_params.npz"
TEXT_COLS = ["feel_text", "food", "soundtrack"]
TOKEN_RE = __import__("re").compile(r"(?u)\b\w\w+\b")


def fill_missing_text(df, columns):
    for col in columns:
        df[col] = df[col].fillna("")


def load_logreg_artifacts(params_path=LOGREG_PARAMS_PATH, tfidf_path=LOGREG_TFIDF_PATH):
    params = np.load(params_path, allow_pickle=True)
    with open(tfidf_path, "r", encoding="utf-8") as f:
        tfidf_meta = json.load(f)

    num_stds = np.asarray(params["num_stds"], dtype=np.float64)
    num_stds = np.where(np.abs(num_stds) < 1e-12, 1.0, num_stds)

    text_specs = [
        (
            col,
            {word: int(idx) for word, idx in tfidf_meta[col]["vocab"].items()},
            np.asarray(tfidf_meta[col]["idf"], dtype=np.float64),
        )
        for col in TEXT_COLS
    ]

    return {
        "coef": np.asarray(params["coef"], dtype=np.float64),
        "intercept": np.asarray(params["intercept"], dtype=np.float64).ravel(),
        "class_order": np.asarray(params["class_order"]),
        "numeric_cols": params["numeric_cols"].tolist(),
        "num_means": np.asarray(params["num_means"], dtype=np.float64),
        "num_stds": num_stds,
        "test_indices": np.asarray(params["test_indices"], dtype=np.int64),
        "text_specs": text_specs,
    }


def load_nb_artifacts(params_path=NB_PARAMS_PATH, vocab_path=NB_VOCAB_PATH):
    params = np.load(params_path, allow_pickle=True)
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab_data = json.load(f)

    return {
        "class_order": np.asarray(params["class_order"]),
        "class_log_prior": np.asarray(params["class_log_prior"], dtype=np.float64),
        "feature_log_prob": np.asarray(params["feature_log_prob"], dtype=np.float64),
        "neg_log_prob": np.asarray(params["neg_log_prob"], dtype=np.float64),
        "vocab_data": vocab_data,
    }


def load_stack_artifacts(path=STACK_META_PATH):
    params = np.load(path, allow_pickle=True)
    return {
        "coef": np.asarray(params["coef"], dtype=np.float64),
        "intercept": np.asarray(params["intercept"], dtype=np.float64).ravel(),
        "class_order": np.asarray(params["class_order"]),
        "bernoulli_class_order": np.asarray(params["bernoulli_class_order"]),
        "logreg_class_order": np.asarray(params["logreg_class_order"]),
    }


def build_binary_matrix(texts, vocab):
    X = np.zeros((len(texts), len(vocab)), dtype=np.float32)
    for i, text in enumerate(texts):
        for word in str(text).split():
            if word in vocab:
                X[i, vocab[word]] = 1.0
    return X


def build_nb_features(df, vocab_data):
    blocks = []
    for col in TEXT_COLS:
        vocab = {word: int(idx) for word, idx in vocab_data[col]["vocab"].items()}
        blocks.append(build_binary_matrix(df[col], vocab))
    return np.hstack(blocks)


def build_tfidf(text_series, vocab, idf_vec):
    block = np.zeros((len(text_series), len(vocab)), dtype=np.float64)

    for i, text in enumerate(text_series):
        tokens = TOKEN_RE.findall(str(text).lower())
        if not tokens:
            continue

        counts = Counter(tok for tok in tokens if tok in vocab)
        if not counts:
            continue

        total = float(sum(counts.values()))
        norm_sq = 0.0

        for tok, count in counts.items():
            j = vocab[tok]
            val = (count / total) * idf_vec[j]
            block[i, j] = val
            norm_sq += val * val

        norm = np.sqrt(norm_sq)
        if norm > 0.0:
            block[i, :] /= norm

    return block


def build_logreg_features(df, numeric_cols, num_means, num_stds, text_specs):
    X_num = df[numeric_cols].to_numpy(dtype=np.float64)
    X_num = np.where(np.isnan(X_num), num_means, X_num)
    X_num = (X_num - num_means) / num_stds

    text_blocks = [build_tfidf(df[col], vocab, idf_vec) for col, vocab, idf_vec in text_specs]
    return np.hstack([X_num] + text_blocks)


def predict_multiclass(X_mat, coef, intercept):
    logits = np.asarray(X_mat @ coef.T) + intercept.reshape(1, -1)
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def predict_bernoulli_nb(X, model):
    log_scores = np.tile(model["class_log_prior"], (X.shape[0], 1))
    log_scores += X @ model["feature_log_prob"].T
    log_scores += (1.0 - X) @ model["neg_log_prob"].T
    return model["class_order"][np.argmax(log_scores, axis=1)]


def align_proba(proba, source_order, target_order):
    source_order = np.asarray(source_order).tolist()
    target_order = np.asarray(target_order).tolist()
    index_lookup = {label: idx for idx, label in enumerate(source_order)}
    return np.column_stack([proba[:, index_lookup[label]] for label in target_order])


logreg_model = load_logreg_artifacts()
nb_model = load_nb_artifacts()
stack_model = load_stack_artifacts()

df = pd.read_csv(DATA_PATH)
fill_missing_text(df, TEXT_COLS)
df = df.loc[logreg_model["test_indices"]].copy()

if "painting" in df.columns:
    painting_names = sorted(df["painting"].dropna().unique())
else:
    painting_names = None

nb_features = build_nb_features(df, nb_model["vocab_data"])
logreg_features = build_logreg_features(
    df,
    logreg_model["numeric_cols"],
    logreg_model["num_means"],
    logreg_model["num_stds"],
    logreg_model["text_specs"],
)

nb_proba = predict_multiclass(
    nb_features,
    nb_model["feature_log_prob"],
    nb_model["class_log_prior"],
)
nb_proba = align_proba(nb_proba, nb_model["class_order"], stack_model["bernoulli_class_order"])

logreg_proba = predict_multiclass(logreg_features, logreg_model["coef"], logreg_model["intercept"])
logreg_proba = align_proba(logreg_proba, logreg_model["class_order"], stack_model["logreg_class_order"])

stack_pred = stack_model["class_order"][np.argmax(predict_multiclass(np.hstack([nb_proba, logreg_proba]), stack_model["coef"], stack_model["intercept"]), axis=1)]

In [ ]:
if "target" in df.columns:
    y_true = df["target"].to_numpy()
    y_pred = np.asarray(stack_pred)
    acc = float(np.mean(y_true == y_pred))
    print(f"Stacked test accuracy: {acc:.4f}")

Stacked test accuracy: 0.9255
